# Fast and slow climate responses: where does the added heat go?
**Week 5 · Surface and deep-ocean boxes**

The forcing-feedback framework:

$$
N'=F+\alpha T_s'.
$$

does not specify where that energy is stored or how quickly the climate adjusts. 

When we looked at the data from the model in the previous lab: **why does part of the warming happen quickly, while another part takes centuries?**

This can be understood by considering a system with two reservoirs:
- **A relatively fast surface component:** the upper ocean and rapidly coupled atmosphere and land surface, represented by one effective temperature and a modest heat capacity.
- **A relatively slow ocean component:** the ocean interior, whose large heat capacity and limited rate of heat exchange give the climate a long memory.

All integration and plotting code is supplied. **Make a guess or work it out, run, then explain.** 

By the end, you should be able to identify fast and slow features in the warming response, explain their connection to storage and exchange, check conservation of energy, and distinguish surface warming from total heat storage. This model describes ocean-driven slow adjustment; it does not represent every slow process in the real climate system, such as ice-sheet or carbon-cycle changes.

## 1. Two temperatures, two budgets

$T_s'$ and $T_d'$ are **warming relative to each box's own reference temperature**, in K. Equal warming does not imply equal absolute temperatures.

$$
Q'=\gamma(T_s'-T_d'),
$$

$$
C_s\frac{dT_s'}{dt}=F+\alpha T_s'-Q',\qquad C_d\frac{dT_d'}{dt}=Q'.
$$

Positive $Q'$ is additional heat transfer from surface to deep ocean. It leaves one box and enters the other. The surface box represents the upper ocean and rapidly coupled surface climate.

| Quantity | Meaning | Units |
|---|---|---|
| $F$ | Constant forcing switched on at year 0 | W m⁻² |
| $\alpha<0$ | Net climate feedback parameter | W m⁻² K⁻¹ |
| $\gamma\geq0$ | Surface–deep exchange coefficient | W m⁻² K⁻¹ |
| $C_s,C_d$ | Surface and deep heat capacities | J m⁻² K⁻¹ |

For an ocean layer of depth $h$, its heat capacity is roughly $C=\rho_wc_wh$, so a deeper layer can hold a lot more heat.

The example uses $F=4$ W m⁻² and $\alpha=-1$ W m⁻² K⁻¹, so $-F/\alpha=4$ K. 

**Can you make a guess or figure it out?** Which box starts warming first? When does the deep box begin gaining energy?

**Your response:**



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from IPython.display import display

%matplotlib inline
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 110})

SECONDS_PER_YEAR = 365.25 * 24 * 3600
BASE = dict(F=4.0, alpha=-1.0, gamma=0.7, Cs=2.2e8, Cd=3.2e9)
SURFACE, DEEP, SINGLE = '#c54b26', '#2374a5', '#686868'
FIGURES = {}

### Supplied numerical model

The function below repeatedly evaluates the two heat budgets and advances the temperatures using SciPy's numerical solver. Its internal time steps are chosen automatically. You do not need to modify this cell.

We display time in **years**. The function also accumulates the incoming energy $\int N'\,dt$ so we can compare it with the energy stored in both boxes.

In [ ]:
def simulate(F=4.0, alpha=-1.0, gamma=0.7, Cs=2.2e8, Cd=3.2e9,
             years=2000.0, output_step=0.5, rtol=1e-8, atol=1e-10):
    """Return temperatures, fluxes, and energy per unit total Earth area."""
    if not all(np.isfinite(v) for v in (F, alpha, gamma, Cs, Cd, years, output_step)):
        raise ValueError('All parameters must be finite.')
    if alpha >= 0 or min(Cs, Cd, years, output_step) <= 0 or gamma < 0:
        raise ValueError('Use negative alpha, positive capacities/time, and nonnegative exchange.')

    def budget(t, state):
        Ts, Td, incoming_GJ = state
        N = F + alpha * Ts
        Q = gamma * (Ts - Td)
        return [SECONDS_PER_YEAR * (N - Q) / Cs,
                SECONDS_PER_YEAR * Q / Cd,
                SECONDS_PER_YEAR * N / 1e9]

    time = np.linspace(0, years, int(np.ceil(years / output_step)) + 1)
    solution = solve_ivp(budget, (0, years), [0.0, 0.0, 0.0],
                         t_eval=time, rtol=rtol, atol=atol)
    if not solution.success:
        raise RuntimeError(solution.message)
    Ts, Td, incoming_GJ = solution.y
    N = F + alpha * Ts
    Q = gamma * (Ts - Td)
    return pd.DataFrame(dict(year=time, Ts=Ts, Td=Td, N=N, Q=Q,
                             surface_storage_rate=N-Q,
                             Es_GJ=Cs*Ts/1e9, Ed_GJ=Cd*Td/1e9,
                             incoming_GJ=incoming_GJ))

two = simulate(**BASE)
one = simulate(**{**BASE, 'gamma': 0.0})
T_equilibrium = -BASE['F'] / BASE['alpha']

def snapshot(run, at_years):
    return pd.DataFrame({name: np.interp(at_years, run.year, run[name])
                         for name in run.columns if name != 'year'},
                        index=pd.Index(at_years, name='Year'))

## 2. Find the fast and slow responses

To contrast what happens if we don't assume multiple boxes (time-scales), we also compare against a one-box. The one-box comparison has **the same forcing, restoring coefficient, and surface heat capacity**; only the exchange is switched off ($\gamma=0$). The disconnected deep reservoir remains at its initial anomaly.

Which surface warms more over the first few decades? Does including the deep ocean change the eventual surface warming? Is 150 years necessarily long enough to reach it?

**Your response:**



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout='constrained')
for ax, end in zip(axes, [50, 2000]):
    for run, field, label, color, style in [
        (one, 'Ts', 'One-box surface', SINGLE, '--'),
        (two, 'Ts', 'Two-box surface', SURFACE, '-'),
        (two, 'Td', 'Deep ocean', DEEP, '-')]:
        part = run[run.year <= end]
        ax.plot(part.year, part[field], label=label, color=color, ls=style, lw=2)
    ax.axhline(T_equilibrium, color='black', ls=':', label='−F / α')
    ax.set(xlim=(0, end), ylim=(0, 4.25), xlabel='Years after forcing',
           ylabel='Temperature anomaly (K)', title=f'First {end:,} years')
axes[0].legend(fontsize=9, loc='center right', bbox_to_anchor=(0.98, 0.33))
FIGURES['warming'] = fig
plt.show()

**Read the curves:** Identify the rapid initial change and the long, slow adjustment in the *surface* curve. Explain why the surface can initially warm quickly and then continue warming slowly for centuries. Why does the deep ocean eventually weaken its cooling influence on the surface? Is it accurate to label the surface curve “the fast response” and the deep curve “the slow response”?

**Your response:**



## 3. Where does the energy go and when

Adding the box budgets gives

$$
\frac{d}{dt}(C_sT_s'+C_dT_d')=N'.
$$

The exchange cancels because it is internal. Starting from zero anomalies,

$$
C_sT_s'+C_dT_d'=\int_0^t N'(u)\,du.
$$

Can the deep ocean store more energy while warming less than the surface? Can the surface keep warming while losing heat through downward exchange?

Before running the model, can you figure out how much heat the two boxes will eventually gain in total? Use their equilibrium warming and heat capacities. Would changing the exchange coefficient change that total, as long as the boxes remain connected?

The left column shows **rates** (W m⁻²). The right shows **accumulated energy** (GJ m⁻²; one GJ is 10⁹ J). The top row shows the first 150 years for rates and 500 years for energy; the bottom row extends both views to 2,000 years.

**Compare the time windows:** As the model approaches equilibrium, what happens to the storage rates? What happens to the accumulated heat?

**Your response:**



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), layout='constrained')
for row, (rate_end, energy_end) in enumerate([(150, 500), (2000, 2000)]):
    rates = two[two.year <= rate_end]
    ax = axes[row, 0]
    ax.plot(rates.year, rates.N, color='black', label="Net incoming N'", lw=2)
    ax.plot(rates.year, rates.Q, color=DEEP, label="Deep storage rate Q'", lw=2)
    ax.plot(rates.year, rates.surface_storage_rate, color=SURFACE,
            label="Surface storage rate N' − Q'", lw=2)
    ax.axhline(0, color='0.7', lw=0.8, zorder=0)
    ax.set(xlabel='Years after forcing', ylabel='Energy storage rate (W/m²)',
           title=f'Storage rates: first {rate_end:,} years', xlim=(0, rate_end))
    ax.legend(fontsize=9)

    energy_run = two[two.year <= energy_end]
    ax = axes[row, 1]
    ax.plot(energy_run.year, energy_run.Es_GJ, color=SURFACE, label='Surface storage', lw=2)
    ax.plot(energy_run.year, energy_run.Ed_GJ, color=DEEP, label='Deep storage', lw=2)
    ax.plot(energy_run.year, energy_run.Es_GJ + energy_run.Ed_GJ,
            color='black', label='Total storage', lw=2)
    ax.plot(energy_run.year, energy_run.incoming_GJ, color='#b57ab4', ls='--',
            label='Accumulated net input', lw=2)
    if row == 1:
        equilibrium_energy = (BASE['Cs'] + BASE['Cd']) * T_equilibrium / 1e9
        ax.axhline(equilibrium_energy, color='0.5', ls=':',
                   label='Equilibrium total storage', lw=1.5)
    ax.set(xlabel='Years after forcing', ylabel='Added energy (GJ/m²)',
           title=f'Accumulated energy: first {energy_end:,} years', xlim=(0, energy_end))
    ax.legend(fontsize=9)
FIGURES['energy'] = fig
plt.show()

residual = two.Es_GJ + two.Ed_GJ - two.incoming_GJ
print(f'Maximum energy-budget residual: {abs(residual).max():.2e} GJ/m²')

## 4. In-class exercise: what controls the response?

### Change the exchange strength ($\gamma$)

Keep forcing, feedback, and both heat capacities fixed. Can you make a guess about how stronger exchange changes surface warming over the first 50 years? Does it change the eventual temperature? Compare the curves below at the same time.

In [ ]:
exchange_values = [0.3, 0.7, 1.5]  # W/m²/K; the baseline is 0.7
exchange_runs = {g: simulate(**{**BASE, 'gamma': g}, years=500)
                 for g in exchange_values}
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
for g, run in exchange_runs.items():
    ax.plot(run.year, run.Ts, label=f'gamma = {g:g}', lw=2)
ax.axhline(T_equilibrium, color='black', ls=':', label='Equilibrium warming')
ax.set(xlabel='Years after forcing', ylabel='Surface anomaly (K)',
       title='Change exchange; keep everything else fixed', xlim=(0, 500))
ax.legend(fontsize=9)
FIGURES['exchange'] = fig
plt.show()

### Change capacity and feedback

Use the two controls at the top of the next code cell. Each run is compared with the original baseline; the baseline itself stays fixed. Keep forcing, deep heat capacity, and exchange unchanged.

1. **Double the surface heat capacity:** set `surface_capacity_factor = 2` and `feedback_factor = 1` (the starting settings below). Can you figure out what happens to the initial warming rate, the eventual temperature, and the eventual accumulated heat?
2. **Make the feedback less restoring:** set `surface_capacity_factor = 1` and `feedback_factor = 0.5`. This changes α from −1 to −0.5 W m⁻² K⁻¹. Which of those three quantities change now?
3. **Try both together:** set `surface_capacity_factor = 2` and `feedback_factor = 0.5`. Can you explain which features come from each change?

Before each run, make a guess or work it out from the budgets. Then run the cell and compare the initial warming, the long-term surface response, and the accumulated heat. The dotted lines show the equilibrium values calculated from the parameters; a curve need not reach its line within the plotted time.

**Your response:**



In [ ]:
# Change these two controls, then rerun this cell for each exercise.
surface_capacity_factor = 2.0
feedback_factor = 1.0

# Positive factors retain positive capacity and negative feedback.
assert surface_capacity_factor > 0 and feedback_factor > 0
experiment_params = {**BASE,
                     'Cs': BASE['Cs'] * surface_capacity_factor,
                     'alpha': BASE['alpha'] * feedback_factor}
experiment = simulate(**experiment_params, years=2000)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), layout='constrained')
comparisons = [(two, BASE, 'Baseline', '#686868'),
               (experiment, experiment_params, 'Your experiment', '#2374a5')]
for run, parameters, label, color in comparisons:
    equilibrium_T = -parameters['F'] / parameters['alpha']
    equilibrium_E = (parameters['Cs'] + parameters['Cd']) * equilibrium_T / 1e9
    early = run[run.year <= 50]
    axes[0].plot(early.year, early.Ts, color=color, label=label, lw=2)
    axes[1].plot(run.year, run.Ts, color=color, label=label, lw=2)
    axes[1].axhline(equilibrium_T, color=color, ls=':', lw=1.5)
    axes[2].plot(run.year, run.Es_GJ + run.Ed_GJ, color=color, label=label, lw=2)
    axes[2].axhline(equilibrium_E, color=color, ls=':', lw=1.5)
for ax in axes:
    ax.set_xlabel('Years after forcing')
    ax.legend(fontsize=9)
axes[0].set(title='First 50 years', ylabel='Surface anomaly (K)', xlim=(0, 50))
axes[1].set(title='Long-term warming', ylabel='Surface anomaly (K)', xlim=(0, 2000))
axes[2].set(title='Total accumulated heat', ylabel='Added energy (GJ/m²)', xlim=(0, 2000))
fig.suptitle(f'Surface capacity × {surface_capacity_factor:g}; feedback × {feedback_factor:g}\n'
             'Dotted lines: equilibrium values', fontsize=12)
FIGURES['parameters'] = fig
plt.show()

**Discuss after running:** Which change affects the initial warming rate? Which changes the eventual temperature? Why can changing a heat capacity change eventual stored heat without changing eventual temperature? Use your combined run to explain the separate roles of capacity and feedback.

**Your response:**



## 5. Return to the Gregory plot

Compare the original one- and two-box time series over 150 years, then plot net incoming radiation against surface warming. Colored markers identify years 1, 10, 50, and 150, using the same colors in both panels.

**Can you make a guess or figure it out?** Will adding the deep reservoir change the Gregory slope or intercept in this model? What changes about the positions reached at the same time?

**Your response:**



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), layout='constrained')
for run, label, style in [(one, 'One box', '--'), (two, 'Two boxes', '-')]:
    part = run[run.year <= 150]
    axes[0].plot(part.year, part.Ts, color=SINGLE if label == 'One box' else SURFACE,
                 ls=style, label=label, lw=2)
x = np.linspace(0, T_equilibrium, 100)
axes[1].plot(x, BASE['F'] + BASE['alpha']*x, color='black', lw=1.5,
             label='Same radiation relation')
mark_times = [1, 10, 50, 150]
colors = ['#6a51a3', '#2171b5', '#238b45', '#cb181d']
for t, color in zip(mark_times, colors):
    for run, marker in [(one, 'o'), (two, 's')]:
        row = snapshot(run, [t]).iloc[0]
        axes[0].scatter(t, row.Ts, color=color, marker=marker, s=45, zorder=4)
        axes[1].scatter(row.Ts, row.N, color=color, marker=marker, s=45, zorder=4,
                        label=f'Year {t}' if marker == 's' else None)
axes[0].set(xlabel='Years after forcing', ylabel='Surface anomaly (K)',
            xlim=(0, 155), title='Different warming histories')
axes[1].set(xlabel='Surface anomaly (K)', ylabel='Net incoming radiation (W/m²)',
            title='Same Gregory line: circles = one, squares = two')
axes[0].legend(fontsize=9)
axes[1].legend(fontsize=9)
FIGURES['gregory_connection'] = fig
plt.show()

## What have we learned?

Complete these statements in your own words:

- A deep reservoir slows surface warming because …
- A slowly changing surface temperature does not establish equilibrium because …
- Internal heat exchange disappears from the combined budget because …
- The Gregory plot alone cannot determine the warming timescale because …

This model resolves two effective heat reservoirs with constant parameters. It does not predict ocean currents, evolving radiative feedbacks, the detailed vertical warming profile, or the ocean's fraction of the whole Earth heat inventory.

The preceding [Gregory activity](gregory_cmip6.ipynb) uses actual CMIP6 output; this notebook isolates the consequences of storage and exchange in an idealized model.

**Your response:**



## Further reading: how researchers use this model

These optional readings connect our numerical experiments to research applications. Start with Held's blog; for the paper and assessment, focus on the physical questions, figures, and conclusions. You do not need to follow the analytical eigenvalue derivations.

### 1. What warming remains if the forcing is removed?

**Isaac Held (2011), [“8. The recalcitrant component of global warming”](https://www.gfdl.noaa.gov/blog_held/8-the-recalcitrant-component-of-global-warming/).** Held uses the same two-box equations to interpret experiments in a comprehensive climate model: abruptly return radiative forcing to its preindustrial value and examine the fast cooling and persistent, slowly evolving warming. The post explains results from [Held et al. (2010)](https://doi.org/10.1175/2009JCLI3466.1).

**Look for:** how the deep ocean gives surface temperature a memory of earlier forcing. Removing the forcing is a different experiment from holding CO₂ constant or stopping emissions. Our notebook currently applies a permanent forcing step; it does not run this removal experiment.

### 2. Why is transient warming smaller than eventual warming?

**Isaac Held (2011), [“4. Transient vs equilibrium climate responses”](https://www.gfdl.noaa.gov/blog_held/3-transient-vs-equilibrium-climate-responses/).** This companion post introduces the surface–deep-ocean model and uses it to distinguish the response while forcing is increasing from the eventual response to fixed forcing. Held writes the restoring coefficient as $\beta$, corresponding to $-\alpha$ in our convention.

**Look for:** why two models with the same equilibrium sensitivity can warm differently during a transient. Connect that argument to our exchange-strength experiments, remembering to specify the time of each comparison.

### 3. Can two boxes reproduce a much more complex model?

**Geoffroy et al. (2013), [“Transient Climate Response in a Two-Layer Energy-Balance Model. Part I: Analytical Solution and Parameter Calibration Using CMIP5 AOGCM Experiments”](https://doi.org/10.1175/JCLI-D-12-00195.1).** The authors calibrate two-layer models using abrupt-4×CO₂ experiments from 16 CMIP5 climate models, then examine their responses under both abrupt and gradually increasing forcing. This is a research example of estimating the capacities and exchange coefficient that we prescribed illustratively.

**Look for:** comparisons between the simple model and the GCM temperature histories. Reproducing an experiment different from the calibration experiment provides a stronger test than fitting the calibration curve alone. The paper also identifies limitations in reproducing the evolving top-of-atmosphere imbalance. Read the abstract and comparison figures first; the analytical derivation is optional background.

### 4. How is this used in a climate assessment?

**IPCC AR6 Working Group I (2021), [Chapter 7, Cross-Chapter Box 7.1: “Physical Emulation of Earth System Models for Scenario Classification and Knowledge Integration in AR6”](https://www.ipcc.ch/report/ar6/wg1/chapter/chapter-7/).** An *emulator* is a computationally inexpensive model calibrated to approximate selected responses of a more complex model or an assessment. This box describes how versions of the two-layer model connect forcing to global temperature; Table 1 lists applications across the report. [Figure 7.7](https://www.ipcc.ch/report/ar6/wg1/figures/chapter-7/figure-7-7/) is a concrete example: estimating the warming contributions of different forcing agents.

**Look for:** what is being predicted and what constrains the parameters. Research and assessment versions can extend our basic equations; our illustrative notebook is not an IPCC-calibrated emulator.